In [1]:
!pip install plotly --quiet
print("Plotly 安装完成！")

Plotly 安装完成！



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'C:\Users\86158\Desktop\project1\online_retail_II.csv', encoding='utf-8')

# 复用之前的清洗逻辑
df = df.dropna(subset=['Customer ID'])
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['Price']
df['Month'] = df['InvoiceDate'].dt.to_period('M').astype(str)
df['Year'] = df['InvoiceDate'].dt.year
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

print(f"数据加载完成！共 {len(df):,} 行")
print(f"时间范围：{df['InvoiceDate'].min().date()} 至 {df['InvoiceDate'].max().date()}")
print(f"国家数量：{df['Country'].nunique()}")
print(f"总销售额：£{df['TotalPrice'].sum():,.0f}")

数据加载完成！共 805,549 行
时间范围：2009-12-01 至 2011-12-09
国家数量：41
总销售额：£17,743,429


In [5]:
monthly = df.groupby('Month').agg(
    销售额=('TotalPrice', 'sum'),
    订单数=('Invoice', 'nunique')
).reset_index()

fig1 = make_subplots(specs=[[{"secondary_y": True}]])
fig1.add_trace(go.Scatter(
    x=monthly['Month'], y=monthly['销售额'],
    name='月销售额(£)', line=dict(color='#2E75B6', width=2.5),
    fill='tozeroy', fillcolor='rgba(46,117,182,0.1)'
), secondary_y=False)
fig1.add_trace(go.Bar(
    x=monthly['Month'], y=monthly['订单数'],
    name='订单数', marker_color='rgba(255,165,0,0.6)'
), secondary_y=True)
fig1.update_layout(
    title='Monthly Sales Revenue & Order Volume',
    xaxis_title='Month', height=420,
    legend=dict(x=0.01, y=0.99),
    plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(tickangle=45)
)
fig1.update_yaxes(title_text="Sales Revenue (£)", secondary_y=False)
fig1.update_yaxes(title_text="Order Count", secondary_y=True)
fig1.show()
print("图1完成")

图1完成


In [6]:
country = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False)
country_top10 = country.head(10).reset_index()
country_top10.columns = ['Country', 'Revenue']

fig2 = px.bar(
    country_top10, x='Revenue', y='Country',
    orientation='h', color='Revenue',
    color_continuous_scale='Blues',
    title='Top 10 Countries by Sales Revenue',
    text=country_top10['Revenue'].apply(lambda x: f'£{x:,.0f}')
)
fig2.update_traces(textposition='outside')
fig2.update_layout(
    height=420, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white',
    yaxis=dict(categoryorder='total ascending')
)
fig2.show()
print("图2完成")

图2完成


In [7]:
top_products = df.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(10).reset_index()
top_products.columns = ['Product', 'Revenue']
top_products['Product'] = top_products['Product'].str[:40]

fig3 = px.bar(
    top_products, x='Revenue', y='Product',
    orientation='h', color='Revenue',
    color_continuous_scale='Greens',
    title='Top 10 Products by Revenue',
    text=top_products['Revenue'].apply(lambda x: f'£{x:,.0f}')
)
fig3.update_traces(textposition='outside')
fig3.update_layout(
    height=420, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white',
    yaxis=dict(categoryorder='total ascending')
)
fig3.show()
print("图3完成")

图3完成


In [8]:
hourly = df.groupby('Hour')['Invoice'].nunique().reset_index()
hourly.columns = ['Hour', 'Orders']

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df.groupby('DayOfWeek')['TotalPrice'].sum().reindex(day_order).reset_index()
daily.columns = ['Day', 'Revenue']

fig4 = make_subplots(rows=1, cols=2,
    subplot_titles=('Orders by Hour of Day', 'Revenue by Day of Week'))

fig4.add_trace(go.Bar(
    x=hourly['Hour'], y=hourly['Orders'],
    marker_color='#2E75B6', name='Orders'
), row=1, col=1)

fig4.add_trace(go.Bar(
    x=daily['Day'], y=daily['Revenue'],
    marker_color='#E8754A', name='Revenue'
), row=1, col=2)

fig4.update_layout(
    height=400, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white',
    title='Purchase Timing Analysis'
)
fig4.show()
print("图4完成")

图4完成


In [9]:
yearly = df.groupby('Year').agg(
    Revenue=('TotalPrice', 'sum'),
    Orders=('Invoice', 'nunique'),
    Customers=('Customer ID', 'nunique')
).reset_index()
yearly['AOV'] = (yearly['Revenue'] / yearly['Orders']).round(2)

fig5 = make_subplots(rows=1, cols=2,
    subplot_titles=('Annual Revenue Comparison', 'Average Order Value by Year'))

fig5.add_trace(go.Bar(
    x=yearly['Year'].astype(str), y=yearly['Revenue'],
    marker_color=['#2E75B6','#1B5E96','#E8754A'],
    text=yearly['Revenue'].apply(lambda x: f'£{x:,.0f}'),
    textposition='outside', name='Revenue'
), row=1, col=1)

fig5.add_trace(go.Bar(
    x=yearly['Year'].astype(str), y=yearly['AOV'],
    marker_color=['#27AE60','#1E8449','#F39C12'],
    text=yearly['AOV'].apply(lambda x: f'£{x:.1f}'),
    textposition='outside', name='AOV'
), row=1, col=2)

fig5.update_layout(
    height=400, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white',
    title='Annual Performance Analysis'
)
fig5.show()
print("图5完成")

图5完成


In [11]:
customer_spend = df.groupby('Customer ID')['TotalPrice'].sum().reset_index()
customer_spend.columns = ['Customer ID', 'Total Spend']
customer_spend_filtered = customer_spend[customer_spend['Total Spend'] < 10000]

fig6 = make_subplots(rows=1, cols=2,
    subplot_titles=('Customer Spend Distribution (< £10,000)', 
                    'Customer Spend Histogram'))

fig6.add_trace(go.Box(
    y=customer_spend_filtered['Total Spend'],
    name='Spend', marker_color='#2E75B6',
    boxpoints='outliers'
), row=1, col=1)

fig6.add_trace(go.Histogram(
    x=customer_spend_filtered['Total Spend'],
    nbinsx=50, marker_color='#2E75B6',
    opacity=0.7, name='Count'
), row=1, col=2)

fig6.update_layout(
    height=420, showlegend=False,
    plot_bgcolor='white', paper_bgcolor='white',
    title='Customer Spend Analysis'
)
fig6.show()
print("图6完成")

图6完成


In [12]:
# 每个客户第一次购买的月份
first_purchase = df.groupby('Customer ID')['InvoiceDate'].min().dt.to_period('M').astype(str)
first_purchase = first_purchase.reset_index()
first_purchase.columns = ['Customer ID', 'FirstMonth']

df2 = df.merge(first_purchase, on='Customer ID')
df2['CustomerType'] = df2.apply(
    lambda x: 'New' if x['Month'] == x['FirstMonth'] else 'Returning', axis=1
)

monthly_type = df2.groupby(['Month','CustomerType'])['Customer ID'].nunique().reset_index()
monthly_type.columns = ['Month','CustomerType','Count']

fig7 = px.bar(
    monthly_type, x='Month', y='Count', color='CustomerType',
    title='Monthly New vs Returning Customers',
    color_discrete_map={'New': '#E8754A', 'Returning': '#2E75B6'},
    barmode='stack'
)
fig7.update_layout(
    height=420, plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(tickangle=45),
    legend=dict(x=0.01, y=0.99)
)
fig7.show()
print("图7完成")

图7完成


In [13]:
product_stats = df.groupby('Description').agg(
    AvgPrice=('Price', 'mean'),
    TotalQty=('Quantity', 'sum'),
    Revenue=('TotalPrice', 'sum')
).reset_index()
product_stats = product_stats[
    (product_stats['AvgPrice'] < 20) & 
    (product_stats['TotalQty'] < 50000) &
    (product_stats['Revenue'] > 500)
]

fig8 = px.scatter(
    product_stats, x='AvgPrice', y='TotalQty',
    size='Revenue', color='Revenue',
    color_continuous_scale='Blues',
    hover_name='Description',
    title='Product Price vs Sales Volume (bubble size = Revenue)',
    labels={'AvgPrice': 'Average Price (£)', 'TotalQty': 'Total Quantity Sold'}
)
fig8.update_layout(
    height=480, plot_bgcolor='white', paper_bgcolor='white'
)
fig8.show()
print("图8完成")

图8完成


In [14]:
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Online Retail Sales Dashboard</title>
    <style>
        body {{ font-family: Arial, sans-serif; background: #f5f7fa; margin: 0; padding: 20px; }}
        h1 {{ text-align: center; color: #1B3A6B; font-size: 28px; margin-bottom: 5px; }}
        p {{ text-align: center; color: #666; margin-bottom: 20px; }}
        .card {{ background: white; border-radius: 10px; padding: 15px;
                 margin-bottom: 20px; box-shadow: 0 2px 8px rgba(0,0,0,0.08); }}
        .kpi-row {{ display: flex; gap: 15px; margin-bottom: 20px; }}
        .kpi {{ flex: 1; background: #1B3A6B; color: white; border-radius: 10px;
                padding: 20px; text-align: center; }}
        .kpi .value {{ font-size: 26px; font-weight: bold; }}
        .kpi .label {{ font-size: 13px; opacity: 0.8; margin-top: 5px; }}
    </style>
</head>
<body>
    <h1>📊 Online Retail Sales Dashboard</h1>
    <p>Data Period: 2009-12-01 to 2011-12-09 &nbsp;|&nbsp; Source: Online Retail II (UCI)</p>

    <div class="kpi-row">
        <div class="kpi">
            <div class="value">£{df['TotalPrice'].sum():,.0f}</div>
            <div class="label">Total Revenue</div>
        </div>
        <div class="kpi">
            <div class="value">{df['Invoice'].nunique():,}</div>
            <div class="label">Total Orders</div>
        </div>
        <div class="kpi">
            <div class="value">{df['Customer ID'].nunique():,}</div>
            <div class="label">Unique Customers</div>
        </div>
        <div class="kpi">
            <div class="value">{df['Country'].nunique()}</div>
            <div class="label">Countries</div>
        </div>
    </div>

    <div class="card">{fig1.to_html(full_html=False, include_plotlyjs='cdn')}</div>
    <div class="card">{fig2.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card">{fig3.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card">{fig4.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card">{fig5.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card">{fig6.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card">{fig7.to_html(full_html=False, include_plotlyjs=False)}</div>
    <div class="card">{fig8.to_html(full_html=False, include_plotlyjs=False)}</div>
</body>
</html>
"""

output_path = r'C:\Users\86158\Desktop\project1\sales_dashboard.html'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_content)

print("8张图看板已导出！")
print(f"路径：{output_path}")

8张图看板已导出！
路径：C:\Users\86158\Desktop\project1\sales_dashboard.html
